In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from datetime import timedelta

In [ ]:
def load_data(filepath):
    """
    Безопасная загрузка .npz файла.
    Возвращает первый массив из архива.
    """
    if not os.path.exists(filepath):
        print(f"ВНИМАНИЕ: Файл не найден {filepath}")
        return None
    with np.load(filepath) as data:
        key = list(data.keys())[0]
        return data[key]

In [ ]:
def get_history_filename(base_date, hours_back, folder="summer"):
    """
    Генерирует имя исторического файла со сдвигом назад во времени.

    Параметры:
    - base_date: datetime объект целевой даты
    - hours_back: количество часов назад
    - folder: папка с данными

    Возвращает:
    - путь к файлу
    """
    past_date = base_date - timedelta(hours=hours_back)
    return os.path.join(folder, f"vars_{past_date.strftime('%Y-%m-%dT%H')}.npz")


In [ ]:
def visualize_error_map(original_data, reconstructed_data, corrupted_data, save_path=None):
    """
    Визуализация карты ошибок на пропущенных ячейках.

    Параметры:
    - original_data: исходные данные (с полными значениями)
    - reconstructed_data: восстановленные данные
    - missing_mask: булева маска пропущенных ячеек (True = пропущено)
    - save_path: если указан, сохраняет график в файл
    """
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    im1 = axes[0].imshow(original_data, aspect='auto', cmap='viridis')
    axes[0].set_title('Данные на первое июля')
    plt.colorbar(im1, ax=axes[0])

    im2 = axes[1].imshow(corrupted_data, aspect='auto', cmap='viridis')
    axes[1].set_title('Повреждённые данные')
    plt.colorbar(im2, ax=axes[1])

    im3 = axes[2].imshow(reconstructed_data, aspect='auto', cmap='viridis')
    axes[2].set_title('Восстановленные данные')
    plt.colorbar(im3, ax=axes[2])

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"График сохранен в {save_path}")

    plt.show()

In [ ]:
def main():
    folder = "summer"
    target_filename = "vars_2022-07-31T00_corrupted.npz"

    target_path = os.path.join(target_filename)
    corrupted_data = load_data(target_path)

    if corrupted_data is None:
        raise FileNotFoundError(f"Целевой файл не найден: {target_path}")

    print(f"Загружен файл: {target_filename}")
    print(f"Размер данных: {corrupted_data.shape}")


    print("\nИспользуется простое заполнение константой 100000")
    reconstructed_data = corrupted_data.copy()
    reconstructed_data[corrupted_data == 0] = 100000.0
    # Чтобы сохранить полученный результат, раскомментируйте следующие 3 строки
    # output_filename = f"restored_{target_filename}"
    # np.savez_compressed(output_filename, data=reconstructed_data)
    # print(f"\nРезультат сохранен в {output_filename}")

    first_07 = load_data(os.path.join(folder, "vars_2022-07-01T00.npz"))
    visualize_error_map(first_07, reconstructed_data, corrupted_data)

In [ ]:
if __name__ == "__main__":
    main()